In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect(r"C:\\Users\\yuki\\Desktop\\qafza's1stProject\\olist.db")

In [3]:
df_orders = pd.read_sql("SELECT * FROM orders", conn)
df_customers = pd.read_sql("SELECT * FROM customers", conn)
df_order_items = pd.read_sql("SELECT * FROM order_items", conn)
df_payments = pd.read_sql("SELECT * FROM order_payments", conn)
df_reviews = pd.read_sql("SELECT * FROM order_reviews", conn)
df_products = pd.read_sql("SELECT * FROM products", conn)
df_sellers = pd.read_sql("SELECT * FROM sellers", conn)
df_geolocation = pd.read_sql("SELECT * FROM geolocation", conn)
df_category_translation = pd.read_sql("SELECT * FROM product_category_name_translation", conn)

In [4]:
tables = {
    "orders": df_orders,
    "customers": df_customers,
    "order_items": df_order_items,
    "order_payments": df_payments,
    "order_reviews": df_reviews,
    "products": df_products,
    "sellers": df_sellers,
    "geolocation": df_geolocation,
    "category_translation": df_category_translation
}

for name, df in tables.items():
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print()

--- orders ---
Shape: (99441, 8)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

--- customers ---
Shape: (99441, 5)
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

--- order_items ---
Shape: (112650, 7)
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

--- order_payments ---
Shape: (103886, 5)
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

--- order_reviews ---
Shape: (99224, 7)
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

--- products ---
Shape: (32951, 9)
Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description

In [5]:
items_agg = df_order_items.groupby("order_id").agg(
    num_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
).reset_index()

# Aggregate order_payments: مجموع قيمة الدفعات + عدد طرق الدفع، لكل order
payments_agg = df_payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    num_payment_methods=("payment_sequential", "count")
).reset_index()

print(items_agg.shape)
print(payments_agg.shape)

(98666, 4)
(99440, 3)


In [6]:
ml_table = df_orders.merge(df_customers, on="customer_id", how="left")
ml_table = ml_table.merge(items_agg, on="order_id", how="left")
ml_table = ml_table.merge(payments_agg, on="order_id", how="left")

print(ml_table.shape)
print(ml_table.columns.tolist())

(99441, 17)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'num_items', 'total_price', 'total_freight', 'total_payment_value', 'num_payment_methods']


In [7]:
ml_table.to_csv("../processedData/ml_table_raw.csv", index=False)
print("Saved successfully!")

Saved successfully!
